# Forecastability Fingerprint Showcase

This walkthrough is the canonical v0.3.1 fingerprint notebook. It uses the prepared
synthetic archetypes from `forecastability.utils.synthetic` and keeps the full story
strictly deterministic: geometry, linear baseline, fingerprint fields, routing, and
the A3 agent-layer explanation all come from reusable package surfaces.

Sections:
- A. What the fingerprint solves
- B. Why four fields instead of disconnected diagnostics
- C. Canonical benchmark series generation
- D. `information_mass` walkthrough
- E. `information_horizon` walkthrough
- F. `information_structure` walkthrough with peak-spacing visuals
- G. `nonlinear_share` walkthrough against the Gaussian-information baseline
- H. Routing walkthrough with caution flags and confidence labels
- I. Agent-layer summary in plain language
- J. Verification and caveats


## Setup

The notebook does not reimplement any science locally. It calls:

- `generate_fingerprint_archetypes()` for the prepared synthetic panel,
- `run_forecastability_fingerprint()` for geometry + fingerprint + routing,
- `compute_linear_information_curve()` for the Gaussian baseline used by `nonlinear_share`,
- `build_fingerprint_showcase_record()` plus the notebook-facing reporting helpers for the
  strict A1/A2/A3 agent summary and reusable figures.


In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from forecastability.reporting.fingerprint_showcase import (
    build_fingerprint_showcase_record,
    build_plain_language_math_summary,
    fingerprint_profile_frame,
    routing_table_frame,
    save_metric_overview,
    save_showcase_profile_grid,
    showcase_summary_frame,
    verify_showcase_records,
    write_frame_csv,
)
from forecastability.services.linear_information_service import compute_linear_information_curve
from forecastability.use_cases.run_forecastability_fingerprint import run_forecastability_fingerprint
from forecastability.utils.synthetic import generate_fingerprint_archetypes

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)

OUTPUT_ROOT = Path("outputs/notebooks/walkthroughs/02_forecastability_fingerprint_showcase")
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_LAG = 24
N_SURROGATES = 99
N = 320


## A — What the fingerprint solves

A raw AMI curve is informative, but it is still harder to communicate than a compact
decision surface. The fingerprint solves that by answering four practical questions in
one object:

- how much usable lag signal remains after surrogate correction,
- how far that signal survives,
- whether the shape decays, repeats, or mixes,
- whether the accepted signal looks mostly linear or needs a richer nonlinear family.

The geometry engine remains the scientific authority. The fingerprint is a deterministic
compression of the corrected geometry, not a second estimator.


## B — Why four fields instead of disconnected diagnostics

The four public fields are designed to stay interpretable together:

- `information_mass`: average amount of accepted corrected AMI,
- `information_horizon`: last accepted horizon,
- `information_structure`: `none`, `monotonic`, `periodic`, or `mixed`,
- `nonlinear_share`: how much accepted AMI sits above the Gaussian-information baseline.

Routing reads those four fields together with `signal_to_noise`, confidence, and caution
flags. The optional agent layer sits downstream and is allowed to explain those values,
but not to modify them.


## C — Canonical benchmark series generation

The prepared panel covers four archetypes that stress the main v0.3.1 branches:

- white noise,
- monotonic AR(1) decay,
- seasonal periodic recurrence,
- nonlinear mixed structure where AMI can exceed the linear baseline.


In [ ]:
series_map = generate_fingerprint_archetypes(n=N, seed=SEED)

records = []
for target_name, series in series_map.items():
    bundle = run_forecastability_fingerprint(
        series,
        target_name=target_name,
        max_lag=MAX_LAG,
        n_surrogates=N_SURROGATES,
        random_state=SEED,
    )
    baseline = compute_linear_information_curve(
        series,
        horizons=[point.horizon for point in bundle.geometry.curve if point.valid],
    )
    records.append(build_fingerprint_showcase_record(bundle=bundle, baseline=baseline))

summary_frame = showcase_summary_frame(records)
routing_frame = routing_table_frame(records)
verification_issues = verify_showcase_records(records)

series_preview = pd.DataFrame({name: series[:8] for name, series in series_map.items()})
series_preview


## D — information_mass walkthrough

`information_mass` answers: after correction and thresholding, how much usable AMI is
left on average over the valid horizon grid? Low mass means either weak lag structure or
structure that collapses under surrogate correction.


In [ ]:
summary_frame[[
    "target_name",
    "signal_to_noise",
    "information_mass",
    "information_structure",
    "agent_summary",
]]


## E — information_horizon walkthrough

`information_horizon` is the last horizon that still clears the acceptance rule. It is
the easiest way to talk about persistence in plain language: how far ahead does any lagged
signal still survive once we stop trusting raw AMI alone?


In [ ]:
summary_frame[["target_name", "information_horizon", "confidence_label", "caution_flags"]]


## F — information_structure walkthrough with peak-spacing visuals

The geometry classifier looks at the corrected profile shape, not at the raw series label.
The figure below overlays three ingredients for each archetype:

- corrected AMI,
- the surrogate threshold `tau`,
- the Gaussian-information baseline used later by `nonlinear_share`.

Accepted horizons are highlighted so the notebook stays visually aligned with the same
mask the routing policy sees.


In [ ]:
profile_path = FIG_DIR / "fingerprint_profiles.png"
save_showcase_profile_grid(records, output_path=profile_path)
write_frame_csv(fingerprint_profile_frame(records[0]), output_path=TABLE_DIR / "white_noise_profile.csv")
Image(filename=str(profile_path))


## G — nonlinear_share walkthrough against Gaussian baseline

`nonlinear_share` is not the same thing as “complexity.” It asks a narrower question:
how much of the accepted corrected AMI still remains after subtracting the information one
would expect from plain linear autocorrelation? High values push routing toward richer
nonlinear families; low values keep the route in linear or seasonal families.


In [ ]:
metric_path = FIG_DIR / "fingerprint_metrics.png"
save_metric_overview(records, output_path=metric_path)
Image(filename=str(metric_path))


## H — Routing walkthrough with caution flags and confidence labels

Routing is deterministic family guidance, not exact-model selection. The comparison table
shows which family groups the policy prefers, what caveats it raises, and how the strict
A3 agent summarizes those results without changing them.


In [ ]:
write_frame_csv(summary_frame, output_path=TABLE_DIR / "fingerprint_summary.csv")
write_frame_csv(routing_frame, output_path=TABLE_DIR / "fingerprint_routing.csv")
routing_frame[[
    "target_name",
    "primary_families",
    "secondary_families",
    "confidence_label",
    "caution_flags",
]]


## I — Agent-layer summary in plain language

The strict deterministic A3 layer is the showcase’s human-facing compression surface. It
reads the same deterministic payload the script writes to JSON and translates the math into
plain language. This keeps the explanation downstream of the numbers, as documented in
[`docs/reference/agent_layer.md`](../../docs/reference/agent_layer.md).

In [ ]:
display(Markdown(build_plain_language_math_summary(records)))


## J — Verification and caveats

Two boundaries matter for v0.3.1:

- the A1/A2/A3 agent surfaces must preserve deterministic geometry, fingerprint, and routing fields,
- the fingerprint showcase remains univariate-first and AMI-first; it is not a multivariate or exact-model-selection claim.

If the list below is empty, the notebook and script are consistent with the deterministic package outputs.


In [ ]:
verification_issues
